# Stanford graph layout figures

This notebook reads the `.out` files generated by `large.cpp`, then saves before/after layout images for the three Stanford network graphs.

The figures intentionally focus on node positions so that the comparison remains readable for very large graphs.

In [ ]:
import os

print(os.getcwd())
os.chdir("../../..")
print(os.getcwd())
assert os.getcwd().endswith("Initial_Placement_for_FR")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("doc/main/large")
OUT_DIR = ROOT / "out"
PLOT_DIR = ROOT / "plot"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

graphs = [
    {
        "stem": "com-amazon.ungraph",
        "name": "Amazon",
        "nodes": 334863,
        "edges": 925872,
    },
    {
        "stem": "com-dblp.ungraph",
        "name": "DBLP",
        "nodes": 317080,
        "edges": 1049866,
    },
    {
        "stem": "roadNet-PA",
        "name": "Pennsylvania road network",
        "nodes": 1088092,
        "edges": 3083796,
    },
]


def load_out(path: Path):
    with path.open("r") as f:
        n, m, k = map(float, f.readline().split())
        n = int(n)
        m = int(m)
        _ = k
        for _ in range(m):
            f.readline()
        positions_size = int(f.readline())
        positions = []
        for _ in range(positions_size):
            values = np.empty((n, 2), dtype=np.float64)
            for i in range(n):
                x, y = map(float, f.readline().split())
                values[i, 0] = x
                values[i, 1] = y
            positions.append(values)
    assert len(positions) == 2
    return n, m, positions


def pick_indices(n: int, limit: int = 200_000):
    if n <= limit:
        return np.arange(n)
    return np.linspace(0, n - 1, limit, dtype=np.int64)


def save_layout_image(positions, title: str, path: Path, limit: int = 200_000):
    points = np.asarray(positions, dtype=np.float64)
    idx = pick_indices(points.shape[0], limit=limit)
    sampled = points[idx]
    min_xy = sampled.min(axis=0)
    max_xy = sampled.max(axis=0)
    center = (min_xy + max_xy) / 2.0
    radius = max(max_xy[0] - min_xy[0], max_xy[1] - min_xy[1]) / 2.0
    radius = max(radius, 1e-9)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(
        sampled[:, 0],
        sampled[:, 1],
        s=0.2,
        c="#1f77b4",
        alpha=0.45,
        linewidths=0,
        rasterized=True,
    )
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(center[0] - 1.03 * radius, center[0] + 1.03 * radius)
    ax.set_ylim(center[1] - 1.03 * radius, center[1] + 1.03 * radius)
    ax.set_axis_off()
    fig.suptitle(title, fontsize=14)
    fig.tight_layout(pad=0.0)
    fig.savefig(path, dpi=300, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)


for graph in graphs:
    out_path = OUT_DIR / f"{graph['stem']}_FR.out"
    n, m, positions = load_out(out_path)
    before_path = PLOT_DIR / f"{graph['stem']}_before.png"
    after_path = PLOT_DIR / f"{graph['stem']}_after.png"
    save_layout_image(positions[0], f"{graph['name']} - before solve_init", before_path)
    save_layout_image(positions[-1], f"{graph['name']} - after solve_init", after_path)
    print(
        f"{graph['stem']}: saved {before_path.name} and {after_path.name} (n={n}, m={m})"
    )